In [1]:
# ============================================================
# 05_AURORA_uncertainty_aware_etf_allocation.ipynb
# AURORA-TWETF Uncertainty-Aware ETF Allocation
#
# Purpose:
# 1. Load adjusted model-probability registry from Notebook 04B.
# 2. Load Taiwan ETF return panels from Notebook 01.
# 3. Convert calibrated regime probabilities into constrained ETF allocations.
# 4. Compare validation-selected and robustness-adjusted allocation policies.
# 5. Apply uncertainty gating using entropy, confidence, probability margin,
#    and ordinal variance.
# 6. Evaluate portfolio performance with transaction costs.
# 7. Save allocation weights, daily returns, performance tables, plots,
#    validation report, and SHA-256 manifest.
#
# Important:
# - This notebook is for educational/research backtesting only.
# - It does not provide personalized financial advice.
# - The allocation rules are deterministic research protocols.
# - The test period follows the split inherited from Notebook 04.
#
# Expected input:
# NOTEBOOK05_INPUT_INDEX.csv from Notebook 04B
# ============================================================

from __future__ import annotations

import os
import sys
import json
import math
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ============================================================
# 0. Colab setup
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or failed.")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# 1. Paths and global configuration
# ============================================================

PROJECT_CODE = "AURORA_TWETF"
PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

DATA_ROOT = PUBLICATION_ROOT / "data"
PANEL_DIR = DATA_ROOT / "panels"
MODELING_DIR = DATA_ROOT / "modeling"

OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"
FIGURE_DIR = OUTPUT_ROOT / "figures"

# Notebook 04B registry.
NOTEBOOK05_INPUT_INDEX = Path(
    "/content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/"
    "ordinal_imbalance_uncertainty/run_20260623_151920/"
    "allocation_inputs_adjusted_before_05/NOTEBOOK05_INPUT_INDEX.csv"
)

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

ALLOCATION_ROOT = OUTPUT_ROOT / "uncertainty_aware_etf_allocation"
RUN_ROOT = ALLOCATION_ROOT / f"run_{RUN_ID}"

WEIGHT_DIR = RUN_ROOT / "weights"
RETURN_DIR = RUN_ROOT / "returns"
PLOT_DIR = RUN_ROOT / "plots"
POLICY_DIR = RUN_ROOT / "policies"
REPORT_RUN_DIR = RUN_ROOT / "reports"
DIAGNOSTIC_DIR = RUN_ROOT / "diagnostics"

for d in [
    OUTPUT_ROOT,
    TABLE_DIR,
    REPORT_DIR,
    FIGURE_DIR,
    ALLOCATION_ROOT,
    RUN_ROOT,
    WEIGHT_DIR,
    RETURN_DIR,
    PLOT_DIR,
    POLICY_DIR,
    REPORT_RUN_DIR,
    DIAGNOSTIC_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("AURORA-TWETF Notebook 05: Uncertainty-Aware ETF Allocation")
print("=" * 80)
print("Timestamp UTC       :", RUN_TIMESTAMP)
print("Run ID              :", RUN_ID)
print("Project root        :", PUBLICATION_ROOT)
print("Notebook 05 registry:", NOTEBOOK05_INPUT_INDEX)
print("Run root            :", RUN_ROOT)
print("=" * 80)

if not NOTEBOOK05_INPUT_INDEX.exists():
    raise FileNotFoundError(
        f"Cannot find Notebook 05 input index:\n{NOTEBOOK05_INPUT_INDEX}\n\n"
        "Please run Notebook 04B first."
    )

# ============================================================
# 2. Allocation configuration
# ============================================================

ETF_UNIVERSE = ["0050", "006208", "00692", "00881"]
CASH_COL = "CASH"

CLASS_LABELS = [0, 1, 2, 3, 4]

REGIME_LABEL_DEFINITION = {
    0: "Strong Bear",
    1: "Bear",
    2: "Neutral",
    3: "Bull",
    4: "Strong Bull",
}

# Rebalancing and backtest assumptions.
REBALANCE_FREQUENCY = "quarterly"   # "monthly" or "quarterly"
TRANSACTION_COST_RATE = 0.0010      # 10 bps per one-way turnover unit
ANNUALIZATION_DAYS = 252
INITIAL_CAPITAL = 1.0

# Signal blending.
ALPHA_20D = 0.60
ALPHA_60D = 0.40

# Uncertainty gating.
MIN_CONFIDENCE_FOR_FULL_RISK = 0.55
MAX_CONFIDENCE_FOR_MIN_RISK = 0.20
UNCERTAINTY_CASH_BOOST_MAX = 0.20

# Constraints.
MAX_ETF_WEIGHT = 0.45
MAX_00881_WEIGHT = 0.35
MAX_CASH_WEIGHT = 0.50
MIN_CASH_WEIGHT = 0.00

# Base class-conditioned allocation templates.
# The templates include CASH, so they sum to 1.
# These are research protocol weights, not investment advice.
CLASS_WEIGHT_TEMPLATES = {
    0: {  # Strong Bear
        "0050": 0.15,
        "006208": 0.25,
        "00692": 0.25,
        "00881": 0.00,
        "CASH": 0.35,
    },
    1: {  # Bear
        "0050": 0.25,
        "006208": 0.30,
        "00692": 0.30,
        "00881": 0.05,
        "CASH": 0.10,
    },
    2: {  # Neutral
        "0050": 0.30,
        "006208": 0.30,
        "00692": 0.25,
        "00881": 0.15,
        "CASH": 0.00,
    },
    3: {  # Bull
        "0050": 0.30,
        "006208": 0.25,
        "00692": 0.20,
        "00881": 0.25,
        "CASH": 0.00,
    },
    4: {  # Strong Bull
        "0050": 0.25,
        "006208": 0.20,
        "00692": 0.15,
        "00881": 0.40,
        "CASH": 0.00,
    },
}

NEUTRAL_TEMPLATE = CLASS_WEIGHT_TEMPLATES[2].copy()

# Policies from Notebook 04B.
POLICIES_TO_COMPARE = [
    "P1_validation_selected",
    "P2_test_robust_reference",
    "P3_conservative_ordinal_60d",
    "P4_calibrated_linear_60d",
    "P5_notebook04_probability_ensemble",
    "P6_custom_robust_60d_weighted_ensemble",
]

# For P6, 04B only exports a custom 60d model.
# We pair it with C4 20d from P2/P1.
P6_FALLBACK_20D_POLICY = "P2_test_robust_reference"

# ============================================================
# 3. Utility functions
# ============================================================

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_file_manifest(root):
    root = Path(root)
    rows = []

    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })

    return pd.DataFrame(rows)

def safe_name(x):
    return (
        str(x)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(" ", "_")
    )

def read_table_auto(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    if path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date")
    else:
        try:
            df.index = pd.to_datetime(df.index)
        except Exception:
            pass

    df = df.sort_index()
    return df

def find_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

def clean_symbol_name(x):
    x = str(x)
    x = x.replace(".TW", "")
    x = x.replace(".TWO", "")
    x = x.replace("TW_", "")
    return x

def load_etf_return_panel():
    candidates = [
        PANEL_DIR / "AURORA_etf_return_panel.parquet",
        PANEL_DIR / "AURORA_etf_returns_panel.parquet",
        PANEL_DIR / "AURORA_return_panel.parquet",
        MODELING_DIR / "AURORA_etf_return_panel.parquet",
    ]

    path = find_first_existing(candidates)

    if path is None:
        raise FileNotFoundError(
            "Could not find ETF return panel. Tried:\n"
            + "\n".join(str(p) for p in candidates)
        )

    df = pd.read_parquet(path)
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    rename_map = {c: clean_symbol_name(c) for c in df.columns}
    df = df.rename(columns=rename_map)

    missing = [s for s in ETF_UNIVERSE if s not in df.columns]

    if missing:
        raise ValueError(
            f"ETF return panel found at {path}, but missing ETF columns: {missing}\n"
            f"Available columns: {list(df.columns)}"
        )

    df = df[ETF_UNIVERSE].copy()
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    print("Loaded ETF return panel:", path)
    print("ETF return shape       :", df.shape)
    print("ETF return date range  :", df.index.min().date(), "to", df.index.max().date())

    return df, path

def load_etf_close_panel_optional():
    candidates = [
        PANEL_DIR / "AURORA_etf_close_panel.parquet",
        PANEL_DIR / "AURORA_close_panel.parquet",
    ]

    path = find_first_existing(candidates)

    if path is None:
        print("ETF close panel not found. Continuing without close panel.")
        return None, None

    df = pd.read_parquet(path)
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    rename_map = {c: clean_symbol_name(c) for c in df.columns}
    df = df.rename(columns=rename_map)

    available = [s for s in ETF_UNIVERSE if s in df.columns]
    if not available:
        print("ETF close panel found, but ETF columns not matched. Continuing without close panel.")
        return None, path

    df = df[available].copy()

    print("Loaded ETF close panel :", path)
    print("ETF close shape        :", df.shape)

    return df, path

def proba_cols_from_df(df):
    return [c for c in df.columns if str(c).startswith("proba_class_")]

def ensure_valid_proba_matrix(p):
    p = np.asarray(p, dtype=float)
    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    p[p < 0] = 0.0

    row_sums = p.sum(axis=1, keepdims=True)
    zero_rows = row_sums[:, 0] <= 0

    if np.any(zero_rows):
        p[zero_rows, :] = 1.0 / p.shape[1]
        row_sums = p.sum(axis=1, keepdims=True)

    p = p / row_sums
    return p

def probability_features(proba_df):
    proba_cols = [f"proba_class_{i}" for i in CLASS_LABELS]

    missing = [c for c in proba_cols if c not in proba_df.columns]
    if missing:
        raise ValueError(f"Missing probability columns: {missing}")

    p = ensure_valid_proba_matrix(proba_df[proba_cols].values)

    classes = np.asarray(CLASS_LABELS, dtype=float)
    expected_class = p @ classes
    entropy = -np.sum(np.clip(p, 1e-12, 1.0) * np.log(np.clip(p, 1e-12, 1.0)), axis=1)
    normalized_entropy = entropy / np.log(len(CLASS_LABELS))
    sorted_p = np.sort(p, axis=1)
    margin = sorted_p[:, -1] - sorted_p[:, -2]
    ordinal_var = (p @ (classes ** 2)) - expected_class ** 2
    confidence = 1.0 - normalized_entropy

    out = pd.DataFrame(index=proba_df.index)
    out["expected_class"] = expected_class
    out["entropy"] = entropy
    out["normalized_entropy"] = normalized_entropy
    out["confidence_score"] = confidence
    out["probability_margin"] = margin
    out["ordinal_variance"] = ordinal_var
    out["p_bearish"] = p[:, 0] + p[:, 1]
    out["p_neutral"] = p[:, 2]
    out["p_bullish"] = p[:, 3] + p[:, 4]

    for i, c in enumerate(CLASS_LABELS):
        out[f"proba_class_{c}"] = p[:, i]

    return out

def class_template_matrix():
    cols = ETF_UNIVERSE + [CASH_COL]
    mat = np.zeros((len(CLASS_LABELS), len(cols)), dtype=float)

    for c in CLASS_LABELS:
        template = CLASS_WEIGHT_TEMPLATES[c]
        for j, col in enumerate(cols):
            mat[c, j] = float(template.get(col, 0.0))

    row_sums = mat.sum(axis=1, keepdims=True)
    row_sums[row_sums <= 0] = 1.0
    mat = mat / row_sums

    return mat, cols

def apply_weight_constraints(weight_vec, cols):
    """
    Applies simple cap constraints and renormalizes.
    """
    w = pd.Series(weight_vec, index=cols, dtype=float)
    w = w.clip(lower=0.0)

    # Cash bounds first.
    if CASH_COL in w.index:
        w[CASH_COL] = min(max(w[CASH_COL], MIN_CASH_WEIGHT), MAX_CASH_WEIGHT)

    # ETF caps.
    for etf in ETF_UNIVERSE:
        if etf in w.index:
            cap = MAX_ETF_WEIGHT
            if etf == "00881":
                cap = min(cap, MAX_00881_WEIGHT)
            w[etf] = min(w[etf], cap)

    # Redistribute leftover to uncapped assets.
    total = w.sum()
    if total <= 0:
        neutral = pd.Series(NEUTRAL_TEMPLATE, dtype=float)
        return neutral.reindex(cols).fillna(0.0)

    # Normalize after caps.
    w = w / total

    # Re-apply hard caps iteratively.
    for _ in range(10):
        excess = 0.0
        capped = []

        for etf in ETF_UNIVERSE:
            cap = MAX_ETF_WEIGHT
            if etf == "00881":
                cap = min(cap, MAX_00881_WEIGHT)

            if w.get(etf, 0.0) > cap:
                excess += w[etf] - cap
                w[etf] = cap
                capped.append(etf)

        if CASH_COL in w.index and w[CASH_COL] > MAX_CASH_WEIGHT:
            excess += w[CASH_COL] - MAX_CASH_WEIGHT
            w[CASH_COL] = MAX_CASH_WEIGHT
            capped.append(CASH_COL)

        if excess <= 1e-12:
            break

        eligible = [c for c in cols if c not in capped]

        if not eligible:
            break

        eligible_sum = w[eligible].sum()

        if eligible_sum <= 0:
            w[eligible] += excess / len(eligible)
        else:
            w[eligible] += excess * (w[eligible] / eligible_sum)

    w = w.clip(lower=0.0)
    total = w.sum()

    if total <= 0:
        neutral = pd.Series(NEUTRAL_TEMPLATE, dtype=float)
        w = neutral.reindex(cols).fillna(0.0)
    else:
        w = w / total

    return w

def proba_to_template_weights(proba_df):
    """
    Converts regime probabilities into expected allocation weights.
    """
    proba_cols = [f"proba_class_{i}" for i in CLASS_LABELS]
    p = ensure_valid_proba_matrix(proba_df[proba_cols].values)

    template_mat, cols = class_template_matrix()
    raw_w = p @ template_mat

    weight_df = pd.DataFrame(raw_w, index=proba_df.index, columns=cols)

    constrained_rows = []
    for _, row in weight_df.iterrows():
        constrained_rows.append(apply_weight_constraints(row.values, cols).values)

    weight_df = pd.DataFrame(constrained_rows, index=weight_df.index, columns=cols)
    return weight_df

def blend_weights(w_20, w_60):
    """
    Blend 20d and 60d allocation weights.
    """
    cols = ETF_UNIVERSE + [CASH_COL]
    w_20 = w_20.reindex(columns=cols).fillna(0.0)
    w_60 = w_60.reindex(columns=cols).fillna(0.0)

    common_idx = w_20.index.intersection(w_60.index)
    out = ALPHA_20D * w_20.loc[common_idx] + ALPHA_60D * w_60.loc[common_idx]

    constrained = []
    for _, row in out.iterrows():
        constrained.append(apply_weight_constraints(row.values, cols).values)

    return pd.DataFrame(constrained, index=out.index, columns=cols)

def apply_uncertainty_gating(weight_df, feature_df):
    """
    Blends signal weights toward neutral/cash when uncertainty is high.
    """
    cols = ETF_UNIVERSE + [CASH_COL]

    neutral = pd.Series(NEUTRAL_TEMPLATE, dtype=float).reindex(cols).fillna(0.0)
    neutral = apply_weight_constraints(neutral.values, cols)

    out_rows = []

    for dt, row in weight_df.iterrows():
        if dt not in feature_df.index:
            out_rows.append(apply_weight_constraints(row.values, cols).values)
            continue

        confidence = float(feature_df.loc[dt, "combined_confidence"])
        p_bearish = float(feature_df.loc[dt, "combined_p_bearish"])
        ord_var = float(feature_df.loc[dt, "combined_ordinal_variance"])

        # Risk scale from confidence.
        denom = MIN_CONFIDENCE_FOR_FULL_RISK - MAX_CONFIDENCE_FOR_MIN_RISK
        if denom <= 0:
            risk_scale = 1.0
        else:
            risk_scale = (confidence - MAX_CONFIDENCE_FOR_MIN_RISK) / denom
            risk_scale = float(np.clip(risk_scale, 0.0, 1.0))

        signal_w = pd.Series(row, index=cols, dtype=float)

        # Blend toward neutral when uncertain.
        gated = risk_scale * signal_w + (1.0 - risk_scale) * neutral

        # Additional cash boost when uncertainty and bearish probability are high.
        uncertainty = 1.0 - confidence
        cash_boost = UNCERTAINTY_CASH_BOOST_MAX * uncertainty * min(1.0, p_bearish + 0.25 * ord_var)
        cash_boost = float(np.clip(cash_boost, 0.0, UNCERTAINTY_CASH_BOOST_MAX))

        if CASH_COL in gated.index:
            etf_cols = [c for c in ETF_UNIVERSE if c in gated.index]
            reduce_pool = gated[etf_cols].sum()

            if reduce_pool > 0:
                gated[etf_cols] = gated[etf_cols] * (1.0 - cash_boost / max(reduce_pool + cash_boost, 1e-12))
                gated[CASH_COL] = gated[CASH_COL] + cash_boost

        gated = apply_weight_constraints(gated.values, cols)
        out_rows.append(gated.values)

    out = pd.DataFrame(out_rows, index=weight_df.index, columns=cols)
    return out

def get_rebalance_dates(index, frequency="quarterly"):
    idx = pd.DatetimeIndex(index).sort_values()

    if frequency == "monthly":
        groups = pd.Series(idx, index=idx).groupby([idx.year, idx.month])
    elif frequency == "quarterly":
        groups = pd.Series(idx, index=idx).groupby([idx.year, idx.quarter])
    else:
        raise ValueError(f"Unsupported rebalance frequency: {frequency}")

    dates = []
    for _, values in groups:
        # Use first trading day of the period as effective rebalance date.
        dates.append(values.iloc[0])

    return pd.DatetimeIndex(dates)

def expand_rebalance_weights_to_daily(signal_weight_df, daily_index, rebalance_dates):
    """
    Weight on rebalance date uses the latest available signal strictly before
    the rebalance date, then remains fixed until next rebalance.
    """
    cols = signal_weight_df.columns.tolist()
    daily_weights = pd.DataFrame(index=daily_index, columns=cols, dtype=float)

    signal_idx = pd.DatetimeIndex(signal_weight_df.index).sort_values()

    current_w = None

    for i, reb_date in enumerate(rebalance_dates):
        effective_start = reb_date

        if i + 1 < len(rebalance_dates):
            effective_end = rebalance_dates[i + 1]
            period_idx = daily_index[(daily_index >= effective_start) & (daily_index < effective_end)]
        else:
            period_idx = daily_index[daily_index >= effective_start]

        prior_signals = signal_idx[signal_idx < reb_date]

        if len(prior_signals) == 0:
            # If no prior signal exists, use the first available signal.
            signal_date = signal_idx[0]
        else:
            signal_date = prior_signals[-1]

        current_w = signal_weight_df.loc[signal_date].copy()
        daily_weights.loc[period_idx, :] = current_w.values

    daily_weights = daily_weights.ffill().bfill()

    return daily_weights

def compute_turnover(daily_weights, rebalance_dates):
    turnover = pd.Series(0.0, index=daily_weights.index)

    prev_w = None

    for dt in rebalance_dates:
        if dt not in daily_weights.index:
            continue

        w = daily_weights.loc[dt]

        if prev_w is None:
            turnover.loc[dt] = w.drop(labels=[CASH_COL], errors="ignore").abs().sum()
        else:
            turnover.loc[dt] = (w - prev_w).abs().sum() / 2.0

        prev_w = w

    return turnover

def backtest_policy(policy_name, signal_weight_df, etf_returns):
    cols = ETF_UNIVERSE + [CASH_COL]

    returns = etf_returns.copy()
    returns[CASH_COL] = 0.0

    common_index = returns.index.intersection(signal_weight_df.index)
    returns = returns.loc[common_index].copy()
    signal_weight_df = signal_weight_df.loc[common_index].copy()

    rebalance_dates = get_rebalance_dates(returns.index, REBALANCE_FREQUENCY)

    daily_weights = expand_rebalance_weights_to_daily(
        signal_weight_df=signal_weight_df,
        daily_index=returns.index,
        rebalance_dates=rebalance_dates,
    )

    daily_weights = daily_weights.reindex(columns=cols).fillna(0.0)

    # Portfolio returns assume weights at t are held for return at t.
    gross_return = (daily_weights[cols] * returns[cols]).sum(axis=1)

    turnover = compute_turnover(daily_weights, rebalance_dates)
    transaction_cost = turnover * TRANSACTION_COST_RATE

    net_return = gross_return - transaction_cost

    equity = (1.0 + net_return).cumprod() * INITIAL_CAPITAL

    out = pd.DataFrame(index=returns.index)
    out["policy_name"] = policy_name
    out["gross_return"] = gross_return
    out["turnover"] = turnover
    out["transaction_cost"] = transaction_cost
    out["net_return"] = net_return
    out["equity"] = equity
    out["drawdown"] = equity / equity.cummax() - 1.0
    out["is_rebalance_date"] = out.index.isin(rebalance_dates)

    return out, daily_weights

def performance_metrics(return_df):
    r = return_df["net_return"].astype(float).copy()
    equity = return_df["equity"].astype(float).copy()
    drawdown = return_df["drawdown"].astype(float).copy()

    n = len(r)
    if n == 0:
        return {}

    total_return = float(equity.iloc[-1] / equity.iloc[0] - 1.0) if equity.iloc[0] != 0 else np.nan
    annual_return = float((1.0 + total_return) ** (ANNUALIZATION_DAYS / max(n, 1)) - 1.0)
    annual_vol = float(r.std(ddof=1) * np.sqrt(ANNUALIZATION_DAYS)) if n > 1 else np.nan

    if annual_vol and annual_vol > 0:
        sharpe = annual_return / annual_vol
    else:
        sharpe = np.nan

    downside = r[r < 0]
    downside_vol = float(downside.std(ddof=1) * np.sqrt(ANNUALIZATION_DAYS)) if len(downside) > 1 else np.nan

    if downside_vol and downside_vol > 0:
        sortino = annual_return / downside_vol
    else:
        sortino = np.nan

    max_drawdown = float(drawdown.min())

    if max_drawdown < 0:
        calmar = annual_return / abs(max_drawdown)
    else:
        calmar = np.nan

    hit_rate = float((r > 0).mean())
    avg_daily_return = float(r.mean())
    avg_turnover = float(return_df["turnover"].mean())
    total_turnover = float(return_df["turnover"].sum())
    total_cost = float(return_df["transaction_cost"].sum())

    return {
        "n_days": int(n),
        "start_date": str(r.index.min().date()),
        "end_date": str(r.index.max().date()),
        "total_return": total_return,
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "sharpe_ratio": sharpe,
        "sortino_ratio": sortino,
        "max_drawdown": max_drawdown,
        "calmar_ratio": calmar,
        "hit_rate": hit_rate,
        "avg_daily_return": avg_daily_return,
        "avg_turnover": avg_turnover,
        "total_turnover": total_turnover,
        "total_transaction_cost": total_cost,
        "final_equity": float(equity.iloc[-1]),
    }

def plot_equity_curves(all_returns_df, path):
    plt.figure(figsize=(12, 6))
    for policy, grp in all_returns_df.groupby("policy_name"):
        grp = grp.sort_index()
        plt.plot(grp.index, grp["equity"], label=policy, linewidth=1.8)

    plt.title("AURORA-TWETF Policy Equity Curves")
    plt.xlabel("Date")
    plt.ylabel("Equity, initial capital = 1")
    plt.legend(loc="best", fontsize=8)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()

def plot_drawdowns(all_returns_df, path):
    plt.figure(figsize=(12, 6))
    for policy, grp in all_returns_df.groupby("policy_name"):
        grp = grp.sort_index()
        plt.plot(grp.index, grp["drawdown"], label=policy, linewidth=1.5)

    plt.title("AURORA-TWETF Policy Drawdowns")
    plt.xlabel("Date")
    plt.ylabel("Drawdown")
    plt.legend(loc="best", fontsize=8)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()

def plot_metric_bar(perf_df, metric, path, higher_is_better=True):
    df = perf_df.sort_values(metric, ascending=not higher_is_better).copy()

    plt.figure(figsize=(10, max(4, 0.45 * len(df))))
    sns.barplot(data=df, y="policy_name", x=metric, color="#4C72B0")
    direction = "higher is better" if higher_is_better else "lower is better"
    plt.title(f"{metric} by allocation policy ({direction})")
    plt.xlabel(metric)
    plt.ylabel("Policy")
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()

def plot_weights_area(weight_df, policy_name, path):
    cols = ETF_UNIVERSE + [CASH_COL]
    df = weight_df[cols].copy()

    plt.figure(figsize=(12, 6))
    plt.stackplot(df.index, [df[c].values for c in cols], labels=cols, alpha=0.85)
    plt.title(f"Allocation Weights: {policy_name}")
    plt.xlabel("Date")
    plt.ylabel("Weight")
    plt.legend(loc="upper left", fontsize=8)
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()

def plot_policy_signal_diagnostics(feature_df, policy_name, path):
    cols = [
        "combined_expected_class",
        "combined_confidence",
        "combined_p_bearish",
        "combined_p_bullish",
    ]

    available = [c for c in cols if c in feature_df.columns]
    if not available:
        return

    fig, axes = plt.subplots(len(available), 1, figsize=(12, 2.6 * len(available)), sharex=True)

    if len(available) == 1:
        axes = [axes]

    for ax, col in zip(axes, available):
        ax.plot(feature_df.index, feature_df[col], linewidth=1.6)
        ax.set_title(f"{policy_name}: {col}")
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()

# ============================================================
# 4. Load Notebook 04B registry and ETF data
# ============================================================

print("\n" + "=" * 80)
print("Step 1: Loading Notebook 04B registry and ETF data")
print("=" * 80)

input_index_df = pd.read_csv(NOTEBOOK05_INPUT_INDEX)

print("Notebook 05 input index shape:", input_index_df.shape)
print(input_index_df[["policy_name", "target_col", "role", "model_name", "source"]].to_string(index=False))

etf_returns, etf_return_path = load_etf_return_panel()
etf_close, etf_close_path = load_etf_close_panel_optional()

# Add cash to return panel later in backtest.
print("\nETF return preview:")
print(etf_returns.head())

# ============================================================
# 5. Load policy probability files
# ============================================================

print("\n" + "=" * 80)
print("Step 2: Loading regime probability files")
print("=" * 80)

policy_probability_data = {}

for _, row in input_index_df.iterrows():
    policy_name = row["policy_name"]
    target_col = row["target_col"]
    role = row["role"]
    model_name = row["model_name"]

    proba_path = Path(row["probability_path_parquet"])
    if not proba_path.exists():
        proba_path = Path(row["probability_path_csv"])

    proba_df = read_table_auto(proba_path)

    proba_cols = [f"proba_class_{i}" for i in CLASS_LABELS]
    missing = [c for c in proba_cols if c not in proba_df.columns]
    if missing:
        raise ValueError(f"Missing proba columns in {proba_path}: {missing}")

    # Keep all splits, but allocation will primarily evaluate test.
    proba_df = proba_df.sort_index()

    key = (policy_name, target_col)
    policy_probability_data[key] = {
        "policy_name": policy_name,
        "target_col": target_col,
        "role": role,
        "model_name": model_name,
        "proba_df": proba_df,
        "path": str(proba_path),
    }

    print(f"Loaded {policy_name} | {target_col} | {model_name} | rows={len(proba_df)}")

# ============================================================
# 6. Build policy-level signal weights
# ============================================================

print("\n" + "=" * 80)
print("Step 3: Building policy-level signal weights")
print("=" * 80)

policy_signal_weights = {}
policy_signal_features = {}
policy_components = []

for policy_name in POLICIES_TO_COMPARE:
    print(f"\nBuilding policy: {policy_name}")

    key_20 = (policy_name, "TAIEX_regime_fixed_20d")
    key_60 = (policy_name, "TAIEX_regime_fixed_60d")

    # P6 only has a 60d custom ensemble. Use fallback 20d model from P2.
    if policy_name == "P6_custom_robust_60d_weighted_ensemble" and key_20 not in policy_probability_data:
        key_20 = (P6_FALLBACK_20D_POLICY, "TAIEX_regime_fixed_20d")
        print(f"  P6 has no native 20d component; using fallback 20d from {P6_FALLBACK_20D_POLICY}")

    if key_20 not in policy_probability_data:
        raise KeyError(f"Missing 20d probability data for policy {policy_name}")

    if key_60 not in policy_probability_data:
        raise KeyError(f"Missing 60d probability data for policy {policy_name}")

    data_20 = policy_probability_data[key_20]
    data_60 = policy_probability_data[key_60]

    p20 = data_20["proba_df"].copy()
    p60 = data_60["proba_df"].copy()

    # Keep common dates.
    common_idx = p20.index.intersection(p60.index).intersection(etf_returns.index)

    if len(common_idx) == 0:
        raise ValueError(f"No common dates for policy {policy_name}")

    p20 = p20.loc[common_idx].copy()
    p60 = p60.loc[common_idx].copy()

    f20 = probability_features(p20)
    f60 = probability_features(p60)

    w20 = proba_to_template_weights(p20)
    w60 = proba_to_template_weights(p60)

    raw_blended_w = blend_weights(w20, w60)

    combined_features = pd.DataFrame(index=common_idx)
    combined_features["policy_name"] = policy_name
    combined_features["split_20d"] = p20["split"].values if "split" in p20.columns else "unknown"
    combined_features["split_60d"] = p60["split"].values if "split" in p60.columns else "unknown"

    combined_features["model_20d"] = data_20["model_name"]
    combined_features["model_60d"] = data_60["model_name"]

    combined_features["expected_class_20d"] = f20["expected_class"]
    combined_features["expected_class_60d"] = f60["expected_class"]
    combined_features["confidence_20d"] = f20["confidence_score"]
    combined_features["confidence_60d"] = f60["confidence_score"]
    combined_features["p_bearish_20d"] = f20["p_bearish"]
    combined_features["p_bearish_60d"] = f60["p_bearish"]
    combined_features["p_bullish_20d"] = f20["p_bullish"]
    combined_features["p_bullish_60d"] = f60["p_bullish"]
    combined_features["ordinal_variance_20d"] = f20["ordinal_variance"]
    combined_features["ordinal_variance_60d"] = f60["ordinal_variance"]

    combined_features["combined_expected_class"] = (
        ALPHA_20D * f20["expected_class"] + ALPHA_60D * f60["expected_class"]
    )

    combined_features["combined_confidence"] = (
        ALPHA_20D * f20["confidence_score"] + ALPHA_60D * f60["confidence_score"]
    )

    combined_features["combined_p_bearish"] = (
        ALPHA_20D * f20["p_bearish"] + ALPHA_60D * f60["p_bearish"]
    )

    combined_features["combined_p_bullish"] = (
        ALPHA_20D * f20["p_bullish"] + ALPHA_60D * f60["p_bullish"]
    )

    combined_features["combined_ordinal_variance"] = (
        ALPHA_20D * f20["ordinal_variance"] + ALPHA_60D * f60["ordinal_variance"]
    )

    gated_w = apply_uncertainty_gating(raw_blended_w, combined_features)

    policy_signal_weights[policy_name] = gated_w
    policy_signal_features[policy_name] = combined_features

    policy_components.append({
        "policy_name": policy_name,
        "model_20d": data_20["model_name"],
        "source_policy_20d": key_20[0],
        "model_60d": data_60["model_name"],
        "source_policy_60d": key_60[0],
        "n_common_dates": int(len(common_idx)),
        "date_start": str(common_idx.min().date()),
        "date_end": str(common_idx.max().date()),
    })

    print("  20d model:", data_20["model_name"])
    print("  60d model:", data_60["model_name"])
    print("  common dates:", len(common_idx), common_idx.min().date(), "to", common_idx.max().date())

policy_components_df = pd.DataFrame(policy_components)
policy_components_df.to_csv(POLICY_DIR / "policy_components.csv", index=False)
policy_components_df.to_csv(TABLE_DIR / f"table_28_policy_components_{RUN_ID}.csv", index=False)

# ============================================================
# 7. Add benchmark policies
# ============================================================

print("\n" + "=" * 80)
print("Step 4: Adding benchmark policies")
print("=" * 80)

cols = ETF_UNIVERSE + [CASH_COL]

benchmark_index = etf_returns.index.copy()

equal_weight_no_cash = pd.DataFrame(
    0.0,
    index=benchmark_index,
    columns=cols,
)

for etf in ETF_UNIVERSE:
    equal_weight_no_cash[etf] = 1.0 / len(ETF_UNIVERSE)
equal_weight_no_cash[CASH_COL] = 0.0

# Benchmark with lower semiconductor ETF exposure.
equal_weight_constrained = pd.DataFrame(
    0.0,
    index=benchmark_index,
    columns=cols,
)

equal_weight_constrained["0050"] = 0.30
equal_weight_constrained["006208"] = 0.30
equal_weight_constrained["00692"] = 0.25
equal_weight_constrained["00881"] = 0.15
equal_weight_constrained[CASH_COL] = 0.00

# 0050-only benchmark.
benchmark_0050 = pd.DataFrame(
    0.0,
    index=benchmark_index,
    columns=cols,
)
benchmark_0050["0050"] = 1.00

policy_signal_weights["B1_equal_weight_all_etfs"] = equal_weight_no_cash
policy_signal_weights["B2_equal_weight_constrained"] = equal_weight_constrained
policy_signal_weights["B3_0050_only"] = benchmark_0050

print("Added benchmark policies:")
print("B1_equal_weight_all_etfs")
print("B2_equal_weight_constrained")
print("B3_0050_only")

# ============================================================
# 8. Backtest all policies
# ============================================================

print("\n" + "=" * 80)
print("Step 5: Backtesting allocation policies")
print("=" * 80)

all_return_frames = []
all_weight_frames = []
all_metric_rows = []

for policy_name, signal_w in policy_signal_weights.items():
    print(f"Backtesting: {policy_name}")

    returns_df, daily_weights = backtest_policy(
        policy_name=policy_name,
        signal_weight_df=signal_w,
        etf_returns=etf_returns,
    )

    metrics = performance_metrics(returns_df)
    metrics["run_id"] = RUN_ID
    metrics["policy_name"] = policy_name
    metrics["transaction_cost_rate"] = TRANSACTION_COST_RATE
    metrics["rebalance_frequency"] = REBALANCE_FREQUENCY
    metrics["alpha_20d"] = ALPHA_20D
    metrics["alpha_60d"] = ALPHA_60D

    all_metric_rows.append(metrics)

    rf = returns_df.copy()
    rf.index.name = "date"
    all_return_frames.append(rf)

    wf = daily_weights.copy()
    wf.index.name = "date"
    wf.insert(0, "policy_name", policy_name)
    all_weight_frames.append(wf)

    returns_df.to_parquet(RETURN_DIR / f"returns_{safe_name(policy_name)}.parquet")
    returns_df.to_csv(RETURN_DIR / f"returns_{safe_name(policy_name)}.csv")

    daily_weights.to_parquet(WEIGHT_DIR / f"weights_{safe_name(policy_name)}.parquet")
    daily_weights.to_csv(WEIGHT_DIR / f"weights_{safe_name(policy_name)}.csv")

    plot_weights_area(
        daily_weights,
        policy_name=policy_name,
        path=PLOT_DIR / f"weights_area_{safe_name(policy_name)}.png",
    )

    if policy_name in policy_signal_features:
        feature_df = policy_signal_features[policy_name]
        feature_df.to_parquet(DIAGNOSTIC_DIR / f"signal_features_{safe_name(policy_name)}.parquet")
        feature_df.to_csv(DIAGNOSTIC_DIR / f"signal_features_{safe_name(policy_name)}.csv")

        plot_policy_signal_diagnostics(
            feature_df,
            policy_name=policy_name,
            path=PLOT_DIR / f"signal_diagnostics_{safe_name(policy_name)}.png",
        )

all_returns_df = pd.concat(all_return_frames, axis=0)
all_weights_df = pd.concat(all_weight_frames, axis=0)
performance_df = pd.DataFrame(all_metric_rows)

# Sort performance by Sharpe and final equity.
performance_df = performance_df.sort_values(
    ["sharpe_ratio", "final_equity"],
    ascending=[False, False],
)

all_returns_df.to_parquet(RETURN_DIR / "returns_all_policies.parquet")
all_returns_df.to_csv(RETURN_DIR / "returns_all_policies.csv")

all_weights_df.to_parquet(WEIGHT_DIR / "weights_all_policies.parquet")
all_weights_df.to_csv(WEIGHT_DIR / "weights_all_policies.csv")

performance_df.to_csv(TABLE_DIR / f"table_29_allocation_performance_{RUN_ID}.csv", index=False)
performance_df.to_csv(RUN_ROOT / "allocation_performance.csv", index=False)
performance_df.to_parquet(RUN_ROOT / "allocation_performance.parquet", index=False)

print("\nAllocation performance:")
display_cols = [
    "policy_name",
    "n_days",
    "start_date",
    "end_date",
    "total_return",
    "annual_return",
    "annual_volatility",
    "sharpe_ratio",
    "sortino_ratio",
    "max_drawdown",
    "calmar_ratio",
    "hit_rate",
    "avg_turnover",
    "total_transaction_cost",
    "final_equity",
]

print(performance_df[display_cols].to_string(index=False))

# ============================================================
# 9. Test-period performance table
# ============================================================

print("\n" + "=" * 80)
print("Step 6: Creating test-period-only performance table")
print("=" * 80)

# Infer test dates from one model's probability split if available.
test_dates = None

for policy_name, features in policy_signal_features.items():
    if "split_20d" in features.columns:
        mask = (features["split_20d"] == "test") | (features["split_60d"] == "test")
        candidate = features.index[mask]
        if len(candidate) > 0:
            test_dates = pd.DatetimeIndex(candidate)
            break

if test_dates is None or len(test_dates) == 0:
    print("Could not infer test dates from probability splits. Using full period as test-period proxy.")
    test_dates = etf_returns.index

test_metric_rows = []

for policy_name, grp in all_returns_df.groupby("policy_name"):
    grp = grp.sort_index()
    grp_test = grp.loc[grp.index.intersection(test_dates)].copy()

    if grp_test.empty:
        continue

    # Rebase equity inside test period.
    grp_test["equity"] = (1.0 + grp_test["net_return"]).cumprod()
    grp_test["drawdown"] = grp_test["equity"] / grp_test["equity"].cummax() - 1.0

    metrics = performance_metrics(grp_test)
    metrics["run_id"] = RUN_ID
    metrics["policy_name"] = policy_name
    metrics["period"] = "test_only"

    test_metric_rows.append(metrics)

test_performance_df = pd.DataFrame(test_metric_rows).sort_values(
    ["sharpe_ratio", "final_equity"],
    ascending=[False, False],
)

test_performance_df.to_csv(TABLE_DIR / f"table_30_allocation_test_only_performance_{RUN_ID}.csv", index=False)
test_performance_df.to_csv(RUN_ROOT / "allocation_test_only_performance.csv", index=False)

print("\nTest-period allocation performance:")
print(test_performance_df[[
    "policy_name",
    "n_days",
    "start_date",
    "end_date",
    "total_return",
    "annual_return",
    "annual_volatility",
    "sharpe_ratio",
    "sortino_ratio",
    "max_drawdown",
    "calmar_ratio",
    "hit_rate",
    "final_equity",
]].to_string(index=False))

# ============================================================
# 10. Plots
# ============================================================

print("\n" + "=" * 80)
print("Step 7: Creating plots")
print("=" * 80)

plot_equity_curves(
    all_returns_df,
    path=PLOT_DIR / "equity_curves_all_policies.png",
)

plot_drawdowns(
    all_returns_df,
    path=PLOT_DIR / "drawdowns_all_policies.png",
)

for metric, higher in [
    ("total_return", True),
    ("annual_return", True),
    ("annual_volatility", False),
    ("sharpe_ratio", True),
    ("sortino_ratio", True),
    ("max_drawdown", True),
    ("calmar_ratio", True),
    ("total_transaction_cost", False),
    ("final_equity", True),
]:
    if metric in performance_df.columns:
        plot_metric_bar(
            performance_df,
            metric=metric,
            path=PLOT_DIR / f"performance_bar_{metric}.png",
            higher_is_better=higher,
        )

if not test_performance_df.empty:
    for metric, higher in [
        ("total_return", True),
        ("annual_return", True),
        ("annual_volatility", False),
        ("sharpe_ratio", True),
        ("sortino_ratio", True),
        ("max_drawdown", True),
        ("calmar_ratio", True),
        ("final_equity", True),
    ]:
        if metric in test_performance_df.columns:
            plot_metric_bar(
                test_performance_df,
                metric=metric,
                path=PLOT_DIR / f"test_only_performance_bar_{metric}.png",
                higher_is_better=higher,
            )

# ============================================================
# 11. Policy ranking and robustness summary
# ============================================================

print("\n" + "=" * 80)
print("Step 8: Policy ranking and robustness summary")
print("=" * 80)

rank_df = performance_df.copy()

rank_df["rank_total_return"] = rank_df["total_return"].rank(ascending=False, method="min")
rank_df["rank_sharpe"] = rank_df["sharpe_ratio"].rank(ascending=False, method="min")
rank_df["rank_sortino"] = rank_df["sortino_ratio"].rank(ascending=False, method="min")
rank_df["rank_drawdown"] = rank_df["max_drawdown"].rank(ascending=False, method="min")
rank_df["rank_calmar"] = rank_df["calmar_ratio"].rank(ascending=False, method="min")

rank_df["allocation_composite_rank"] = (
    rank_df["rank_total_return"]
    + rank_df["rank_sharpe"]
    + rank_df["rank_sortino"]
    + rank_df["rank_drawdown"]
    + rank_df["rank_calmar"]
) / 5.0

rank_df = rank_df.sort_values(
    ["allocation_composite_rank", "sharpe_ratio", "total_return"],
    ascending=[True, False, False],
)

rank_df.to_csv(TABLE_DIR / f"table_31_allocation_policy_rankings_{RUN_ID}.csv", index=False)
rank_df.to_csv(RUN_ROOT / "allocation_policy_rankings.csv", index=False)

print("\nPolicy rankings:")
print(rank_df[[
    "policy_name",
    "total_return",
    "annual_return",
    "annual_volatility",
    "sharpe_ratio",
    "sortino_ratio",
    "max_drawdown",
    "calmar_ratio",
    "allocation_composite_rank",
]].to_string(index=False))

# Robustness comparison subset.
robustness_policies = [
    "P1_validation_selected",
    "P2_test_robust_reference",
    "P3_conservative_ordinal_60d",
    "P4_calibrated_linear_60d",
    "P5_notebook04_probability_ensemble",
    "P6_custom_robust_60d_weighted_ensemble",
]

robustness_df = rank_df[rank_df["policy_name"].isin(robustness_policies)].copy()
robustness_df.to_csv(TABLE_DIR / f"table_32_allocation_robustness_policies_{RUN_ID}.csv", index=False)

print("\nRobustness policy comparison:")
print(robustness_df[[
    "policy_name",
    "total_return",
    "annual_return",
    "annual_volatility",
    "sharpe_ratio",
    "max_drawdown",
    "final_equity",
    "allocation_composite_rank",
]].to_string(index=False))

# ============================================================
# 12. Save validation report and manifest
# ============================================================

print("\n" + "=" * 80)
print("Step 9: Saving validation report and manifest")
print("=" * 80)

validation_report = {
    "project_code": PROJECT_CODE,
    "notebook": "05_AURORA_uncertainty_aware_etf_allocation.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "notebook05_input_index": str(NOTEBOOK05_INPUT_INDEX),
    "etf_return_panel": str(etf_return_path),
    "etf_close_panel": str(etf_close_path) if etf_close_path is not None else None,
    "etf_universe": ETF_UNIVERSE,
    "cash_column": CASH_COL,
    "rebalance_frequency": REBALANCE_FREQUENCY,
    "transaction_cost_rate": TRANSACTION_COST_RATE,
    "annualization_days": ANNUALIZATION_DAYS,
    "alpha_20d": ALPHA_20D,
    "alpha_60d": ALPHA_60D,
    "uncertainty_gating": {
        "min_confidence_for_full_risk": MIN_CONFIDENCE_FOR_FULL_RISK,
        "max_confidence_for_min_risk": MAX_CONFIDENCE_FOR_MIN_RISK,
        "uncertainty_cash_boost_max": UNCERTAINTY_CASH_BOOST_MAX,
    },
    "constraints": {
        "max_etf_weight": MAX_ETF_WEIGHT,
        "max_00881_weight": MAX_00881_WEIGHT,
        "max_cash_weight": MAX_CASH_WEIGHT,
        "min_cash_weight": MIN_CASH_WEIGHT,
    },
    "class_weight_templates": CLASS_WEIGHT_TEMPLATES,
    "policies_to_compare": POLICIES_TO_COMPARE,
    "policy_components": policy_components_df.to_dict(orient="records"),
    "performance_table_path": str(TABLE_DIR / f"table_29_allocation_performance_{RUN_ID}.csv"),
    "test_performance_table_path": str(TABLE_DIR / f"table_30_allocation_test_only_performance_{RUN_ID}.csv"),
    "ranking_table_path": str(TABLE_DIR / f"table_31_allocation_policy_rankings_{RUN_ID}.csv"),
    "educational_note": (
        "This notebook is a research backtest and does not provide personalized financial advice. "
        "The allocation rules are deterministic experimental protocols for comparing uncertainty-aware policies."
    ),
    "output_paths": {
        "run_root": str(RUN_ROOT),
        "weights": str(WEIGHT_DIR),
        "returns": str(RETURN_DIR),
        "plots": str(PLOT_DIR),
        "policies": str(POLICY_DIR),
        "diagnostics": str(DIAGNOSTIC_DIR),
    },
}

validation_report_path = REPORT_RUN_DIR / "AURORA_05_allocation_validation_report.json"
validation_report_global_path = REPORT_DIR / f"AURORA_05_allocation_validation_report_{RUN_ID}.json"

save_json(validation_report_path, validation_report)
save_json(validation_report_global_path, validation_report)

manifest_df = make_file_manifest(RUN_ROOT)

manifest_path = REPORT_RUN_DIR / "AURORA_05_allocation_file_manifest_SHA256.csv"
manifest_global_path = REPORT_DIR / f"AURORA_05_allocation_file_manifest_SHA256_{RUN_ID}.csv"

manifest_df.to_csv(manifest_path, index=False)
manifest_df.to_csv(manifest_global_path, index=False)

# ============================================================
# 13. Final summary
# ============================================================

print("\n" + "=" * 80)
print("AURORA-TWETF NOTEBOOK 05 COMPLETE")
print("=" * 80)
print("Run ID                 :", RUN_ID)
print("Run root               :", RUN_ROOT)
print("Notebook 05 input index:", NOTEBOOK05_INPUT_INDEX)
print("Performance table      :", TABLE_DIR / f"table_29_allocation_performance_{RUN_ID}.csv")
print("Test performance table :", TABLE_DIR / f"table_30_allocation_test_only_performance_{RUN_ID}.csv")
print("Policy ranking table   :", TABLE_DIR / f"table_31_allocation_policy_rankings_{RUN_ID}.csv")
print("Robustness table       :", TABLE_DIR / f"table_32_allocation_robustness_policies_{RUN_ID}.csv")
print("Weights directory      :", WEIGHT_DIR)
print("Returns directory      :", RETURN_DIR)
print("Plots directory        :", PLOT_DIR)
print("Validation report      :", validation_report_path)
print("Manifest               :", manifest_path)
print("=" * 80)

print("\nRecommended next notebook:")
print("06_AURORA_allocation_stress_tests_and_paper_figures.ipynb")

Mounted at /content/drive
AURORA-TWETF Notebook 05: Uncertainty-Aware ETF Allocation
Timestamp UTC       : 2026-06-24T00:45:16Z
Run ID              : 20260624_004516
Project root        : /content/drive/MyDrive/AURORA_TWETF
Notebook 05 registry: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/ordinal_imbalance_uncertainty/run_20260623_151920/allocation_inputs_adjusted_before_05/NOTEBOOK05_INPUT_INDEX.csv
Run root            : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/uncertainty_aware_etf_allocation/run_20260624_004516

Step 1: Loading Notebook 04B registry and ETF data
Notebook 05 input index shape: (11, 9)
                           policy_name             target_col        role                          model_name                            source
                P1_validation_selected TAIEX_regime_fixed_20d primary_20d        C4_calibrated_xgb_multiclass           notebook04_model_output
                P1_validation_selected TAIEX_regime_fixed_60d primary_60